In [1]:
# -*- coding: utf-8 -*-
"""
改进版 ST-TransUNet + 格点 PCMCI
（m1固定无参数相加 / m4局地Cross-Attention）

核心设计
--------
1. m1 是严格的无门控、无额外可学习参数相加版本：
   - 只使用原始 MCI 场；
   - 多个 event_type 在 event 维取算术平均；
   - 保留 predictor / lag / latitude / longitude 的一一对应；
   - 直接在输入层执行 vx + fixed_pcmci_mci；
   - 不使用 PCMCI Encoder、prior gate、fusion gate、LayerNorm、
     affine scale/bias、逐格点 residual 或任何可学习融合权重。
2. m4 保持原来的局地 Cross-Attention 结构：
   - mci/qval/signif 作为连续输入；
   - 每个 lag 经独立 PCMCI Encoder；
   - 使用 prior gate、局地 Cross-Attention、fusion gate 和 LayerNorm；
   - 可选 affine / affine_residual 校准及 L2 约束。
3. 为保证物理对应，默认严格要求：
   - PCMCI predictor 数 == vx 输入通道数；
   - PCMCI lag 数 == timestep；
   - PCMCI latitude/longitude 尺寸 == vx 空间尺寸。

输入约定
--------
vx: (time, latitude, longitude, predictor)
vy: 回归/二分类可为 (time, latitude, longitude) 或
    (time, latitude, longitude, output_channel)
PCMCI: (event_type, predictor, lag, latitude, longitude)

返回值保持原接口：
model, predicty, testy, r, p, heatmap, weights
"""


# 版本说明：
# - pcmci_mode='m1'：
#   fixed_pcmci = mean_event(mci)，然后在输入层固定相加：
#   x = x + fixed_pcmci。
#   m1 不建立任何额外可学习 PCMCI 模块，也不使用 gate。
# - pcmci_mode='m4'及其兼容别名：
#   保持门控局地 Cross-Attention 残差融合。

def Auto_ST_TransUnet_pytorch(
    vy,
    vx,
    timestep,
    downnum,
    covnum,
    test_size=0.2,
    valid_size=0.1,
    k_fold=None,
    task_mode='regression',
    if_best_mode='no',
    modelpath=None,
    ifrandom_split='yes',
    layer_time=2,
    baselayer=64,
    cov_kernelsize=3,
    cov_strides=1,
    pool_method='maxpooling',
    pool_kernel_size=2,
    pool_strides=2,
    ifnormalization='no',
    normalization_method=None,
    trans_num=2,
    num_heads=2,
    dim_feedforward=256,
    trans_dropout_rate=0.0,
    activate='tanh',
    nlp_activate='relu',
    if_last_act='no',
    pcmci_mode='none',
    pcmci_mci=None,
    pcmci_qval=None,
    pcmci_signif=None,
    pcmci_q_th=0.1,
    pcmci_use_abs=False,
    if_print_model='yes',
    loss_function='default',
    optimizer='SGD',
    metrics='default',
    learning_rate=0.01,
    epochs=2000,
    batch_size=20,
    if_early_stopping=None,
    ifheatmap='no',
    ifweight='yes',
    ifmute='no',
    ifsave='no',
    savepath=None,
    device='cpu',
    # -------- 新增参数：均放在末尾，不破坏原调用 --------
    pcmci_lag_order='as_is',
    pcmci_trainable_mode='affine_residual',
    pcmci_reg_lambda=1e-5,
    pcmci_channel_align='strict',
    pcmci_spatial_align='strict',
    pcmci_gate_per_channel=False,
    pcmci_store_diagnostics=False,
    num_workers=0,
    random_state=25,
    temporal_transformer_mode='local',
):
    """
    改进版 ST-TransUNet + 格点 PCMCI。

    新增参数
    --------
    pcmci_mode : {'none', 'm1', 'm4', 'm5', 'local',
                  'local_cross_attention', 'local_gated_cross_attention'}
        none/no：不使用 PCMCI。
        m1：固定无参数相加。只使用原始 MCI，并在 event_type 维取算术平均，
        得到与 vx 的 (T,C,H,W) 完全对应的固定场，然后执行
        ``x_input = x_input + mean_event(mci)``。m1 不使用 PCMCI Encoder、
        prior gate、fusion gate、LayerNorm、affine/residual 校准或其他
        可学习融合参数。
        m4/m5/local/local_cross_attention/local_gated_cross_attention：保持原有
        局地 Cross-Attention 融合流程。

    pcmci_lag_order : {'as_is', 'reverse'}
        'as_is'：PCMCI lag 维保持原顺序。
        'reverse'：翻转 PCMCI lag 维。
        若 vx 时间窗按 [t-2, t-1, t] 排列，而 PCMCI lag 按 [0,1,2]
        排列，通常应设置为 'reverse'，使 lag=2 对应最早输入、lag=0
        对应最后输入。必须结合 nc 文件中的 lag 坐标确认。

    pcmci_trainable_mode : {'encoder', 'affine', 'affine_residual'}
        仅对 m4 类模式生效；m1 会忽略该参数并固定使用原始 MCI。
        encoder：原始 PCMCI 为固定 buffer，但 PCMCI Encoder/attention/gate 可训练。
        affine：增加按 field/event/predictor/lag 的可训练 scale 和 bias。
        affine_residual：在 affine 基础上，再增加逐格点可训练 residual；
        自由度最高，需用 pcmci_reg_lambda 约束。

    pcmci_reg_lambda : float
        PCMCI 可训练校准参数的 L2 约束系数。

    pcmci_channel_align : {'strict', 'pad_or_truncate'}
        strict：PCMCI predictor 数必须与 vx 通道数一致。
        pad_or_truncate：不一致时在 predictor 维补默认值或截断。
        为避免变量顺序错位，建议保持 strict。

    pcmci_spatial_align : {'strict', 'interpolate'}
        strict：PCMCI 与 vx 原始 H/W 必须一致。
        interpolate：使用双线性插值到 vx H/W。

    pcmci_gate_per_channel : bool
        仅对 m4 类模式生效。
        False：gate 形状为每个 lag/格点一个标量。
        True：gate 形状为每个 lag/格点/特征通道一个值。

    pcmci_store_diagnostics : bool
        True 时保存最近一次 forward 的诊断结果。
        m4 保存平均 prior gate、fusion gate 和 attention 权重；
        m1 仅保存实际固定相加的 direct_add_map。
        可通过 model.get_pcmci_diagnostics() 获取。

    temporal_transformer_mode : {'local', 'global_flatten'}
        local：推荐。每个 bottleneck 格点独立沿时间维做 Transformer，
        d_model=C_bottleneck，保持 latitude/longitude 对应且参数量小。
        global_flatten：兼容旧版，把 C*H*W 展平为每个时间 token；
        参数量可能极大，不推荐。
    """

    import copy
    import datetime
    import math
    import os
    import warnings

    import numpy as np
    import torch
    import torch.nn.functional as F
    from scipy.stats import pearsonr
    from sklearn.model_selection import KFold, train_test_split
    from torch import nn
    from torch.optim import Adam, NAdam, SGD
    from torch.utils.data import DataLoader, TensorDataset

    warnings.filterwarnings('ignore')

    # ============================================================
    # 0. 基础检查与设备
    # ============================================================
    if not isinstance(timestep, int) or timestep < 1:
        raise ValueError(f"timestep 必须为正整数，当前为 {timestep!r}")
    if not isinstance(downnum, int) or downnum < 1:
        raise ValueError(f"downnum 必须为正整数，当前为 {downnum!r}")
    if not isinstance(covnum, int) or covnum < 1:
        raise ValueError(f"covnum 必须为正整数，当前为 {covnum!r}")
    if cov_strides != 1:
        raise ValueError(
            "该改进版为保证 vx 与 PCMCI 编码后的格点严格对应，"
            "目前要求 cov_strides=1；空间降采样统一由 pooling 完成。"
        )
    if pool_kernel_size != pool_strides:
        raise ValueError(
            "为保证 Encoder/Decoder 空间对应，要求 pool_kernel_size "
            "与 pool_strides 相同。"
        )
    if task_mode not in {'regression', 'binary_classify', 'multi_classify'}:
        raise ValueError(f"不支持的 task_mode={task_mode!r}")

    def _parse_device(dev):
        if isinstance(dev, torch.device):
            out = dev
        elif isinstance(dev, int):
            out = torch.device(f'cuda:{dev}')
        elif isinstance(dev, str):
            s = dev.strip().lower()
            if s == 'gpu':
                s = 'cuda:0'
            out = torch.device(s)
        else:
            raise TypeError(f"device 类型错误：{type(dev)}")

        if out.type == 'cuda':
            if not torch.cuda.is_available():
                raise RuntimeError("当前 PyTorch 无法使用 CUDA。")
            index = 0 if out.index is None else out.index
            if index >= torch.cuda.device_count():
                raise RuntimeError(
                    f"指定了 cuda:{index}，但当前进程仅识别到 "
                    f"{torch.cuda.device_count()} 张 GPU；"
                    f"CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')}"
                )
            out = torch.device(f'cuda:{index}')
        return out

    devices = _parse_device(device)
    print('=' * 72)
    print('实际使用设备：', devices)
    if devices.type == 'cuda':
        print('GPU 名称：', torch.cuda.get_device_name(devices))
        print('PyTorch 可见 GPU 数：', torch.cuda.device_count())
        print('CUDA_VISIBLE_DEVICES：', os.environ.get('CUDA_VISIBLE_DEVICES'))
    print('=' * 72)

    np.random.seed(random_state)
    torch.manual_seed(random_state)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(random_state)

    # ============================================================
    # 1. 数据整理
    # ============================================================
    vx_np = np.asarray(vx)
    vy_np = np.asarray(vy)

    if vx_np.ndim != 4:
        raise ValueError(
            "vx 必须为 (time, latitude, longitude, predictor) 四维数组，"
            f"当前 shape={vx_np.shape}"
        )
    if vx_np.shape[0] < timestep:
        raise ValueError(
            f"vx 时间长度 {vx_np.shape[0]} 小于 timestep={timestep}"
        )

    vx_np = np.nan_to_num(vx_np, nan=0.0, posinf=0.0, neginf=0.0).astype(
        np.float32, copy=False
    )

    n_time, input_h, input_w, input_channels = vx_np.shape
    n_seq = n_time - timestep + 1

    # 使用 float32，并按时间位置批量赋值，避免原代码默认 float64。
    vxs = np.empty(
        (n_seq, timestep, input_channels, input_h, input_w),
        dtype=np.float32,
    )
    for lag_index in range(timestep):
        vxs[:, lag_index] = vx_np[
            lag_index:lag_index + n_seq
        ].transpose(0, 3, 1, 2)

    if task_mode in {'regression', 'binary_classify'}:
        if vy_np.ndim == 3:
            vy_np = vy_np[..., None]
        if vy_np.ndim != 4:
            raise ValueError(
                "回归/二分类 vy 必须是 (time,H,W) 或 (time,H,W,C_out)，"
                f"当前 shape={vy_np.shape}"
            )
        if vy_np.shape[:3] != (n_time, input_h, input_w):
            raise ValueError(
                "vy 与 vx 的 time/H/W 不一致："
                f"vx={vx_np.shape[:3]}, vy={vy_np.shape[:3]}"
            )
        vys = np.nan_to_num(
            vy_np[timestep - 1:], nan=0.0, posinf=0.0, neginf=0.0
        ).astype(np.float32, copy=False).transpose(0, 3, 1, 2)
        output_channels = int(vys.shape[1])
        num_classes = None
    else:
        # 多分类标签统一为 (N,H,W)，类别值为整数。
        if vy_np.ndim == 4 and vy_np.shape[-1] == 1:
            vy_np = vy_np[..., 0]
        if vy_np.ndim != 3:
            raise ValueError(
                "多分类 vy 必须是 (time,H,W) 或 (time,H,W,1)，"
                f"当前 shape={vy_np.shape}"
            )
        if vy_np.shape != (n_time, input_h, input_w):
            raise ValueError(
                f"vy shape={vy_np.shape} 与 vx 的 time/H/W 不一致"
            )
        vys = vy_np[timestep - 1:].astype(np.int64, copy=False)
        valid_labels = vys[vys != -100]
        if valid_labels.size == 0:
            raise ValueError("多分类标签全部为 ignore_index=-100。")
        num_classes = int(np.max(valid_labels)) + 1
        output_channels = num_classes

    # ============================================================
    # 2. PCMCI 预处理；不做显著性硬过滤
    # ============================================================
    use_pcmci = pcmci_mode not in {None, 'none', 'no'}
    local_pcmci_modes = {
        'm1', 'm4', 'm5', 'local', 'local_cross_attention',
        'local_gated_cross_attention'
    }
    if use_pcmci and pcmci_mode not in local_pcmci_modes:
        raise ValueError(
            f"本改进版支持 pcmci_mode='none' 或模式 {sorted(local_pcmci_modes)}；"
            f"当前为 {pcmci_mode!r}。"
        )

    pcmci_fields_np = None
    pcmci_m1_add_np = None
    pcmci_shape_info = None

    def _normalize_pcmci_5d(arr, name, fill_value):
        if arr is None:
            return None
        arr = np.asarray(arr)
        if arr.ndim == 4:
            # (C,L,H,W) -> (1,C,L,H,W)
            arr = arr[None, ...]
        if arr.ndim != 5:
            raise ValueError(
                f"{name} 必须为 (E,C,L,H,W) 或 (C,L,H,W)，"
                f"当前 shape={arr.shape}"
            )
        return np.nan_to_num(
            arr, nan=fill_value, posinf=fill_value, neginf=fill_value
        ).astype(np.float32, copy=False)

    if use_pcmci:
        mci_np = _normalize_pcmci_5d(pcmci_mci, 'pcmci_mci', 0.0)
        if mci_np is None:
            raise ValueError("启用 PCMCI 时 pcmci_mci 不能为空。")

        qval_np = _normalize_pcmci_5d(pcmci_qval, 'pcmci_qval', 1.0)
        signif_np = _normalize_pcmci_5d(pcmci_signif, 'pcmci_signif', 0.0)

        if qval_np is None:
            qval_np = np.ones_like(mci_np, dtype=np.float32)
        if signif_np is None:
            # 不进行 q < threshold 的硬二值化；使用连续置信度 1-q。
            signif_np = 1.0 - np.clip(qval_np, 0.0, 1.0)

        if qval_np.shape != mci_np.shape:
            raise ValueError(
                f"qval 与 mci 形状不一致：{qval_np.shape} != {mci_np.shape}"
            )
        if signif_np.shape != mci_np.shape:
            raise ValueError(
                f"signif 与 mci 形状不一致："
                f"{signif_np.shape} != {mci_np.shape}"
            )

        event_num, pcmci_predictors, pcmci_lags, pcmci_h, pcmci_w = mci_np.shape

        # predictor 对齐。推荐 strict，避免不同变量被错误对应。
        if pcmci_predictors != input_channels:
            if pcmci_channel_align == 'strict':
                raise ValueError(
                    "PCMCI predictor 数必须与 vx 输入通道数一致，且顺序必须一致："
                    f"PCMCI={pcmci_predictors}, vx={input_channels}。"
                )
            if pcmci_channel_align != 'pad_or_truncate':
                raise ValueError(
                    "pcmci_channel_align 仅支持 'strict' 或 'pad_or_truncate'。"
                )

            def _align_predictors(arr, target, fill):
                current = arr.shape[1]
                if current > target:
                    return arr[:, :target, ...]
                if current < target:
                    pad_shape = list(arr.shape)
                    pad_shape[1] = target - current
                    pad = np.full(pad_shape, fill, dtype=np.float32)
                    return np.concatenate([arr, pad], axis=1)
                return arr

            mci_np = _align_predictors(mci_np, input_channels, 0.0)
            qval_np = _align_predictors(qval_np, input_channels, 1.0)
            signif_np = _align_predictors(signif_np, input_channels, 0.0)
            pcmci_predictors = input_channels

        # lag 必须严格对应时间步。
        if pcmci_lags != timestep:
            raise ValueError(
                "为保证逐 lag 对应，PCMCI lag 数必须等于 timestep："
                f"PCMCI lag={pcmci_lags}, timestep={timestep}。"
            )

        if pcmci_lag_order == 'reverse':
            mci_np = mci_np[:, :, ::-1, :, :].copy()
            qval_np = qval_np[:, :, ::-1, :, :].copy()
            signif_np = signif_np[:, :, ::-1, :, :].copy()
        elif pcmci_lag_order != 'as_is':
            raise ValueError("pcmci_lag_order 仅支持 'as_is' 或 'reverse'。")

        # 空间对应。
        if (pcmci_h, pcmci_w) != (input_h, input_w):
            if pcmci_spatial_align == 'strict':
                raise ValueError(
                    "PCMCI 与 vx 的原始空间尺寸必须一致："
                    f"PCMCI={(pcmci_h, pcmci_w)}, vx={(input_h, input_w)}"
                )
            if pcmci_spatial_align != 'interpolate':
                raise ValueError(
                    "pcmci_spatial_align 仅支持 'strict' 或 'interpolate'。"
                )
            # 只在用户明确要求时插值；E/C/L 展成 batch。
            def _resize_pcmci(arr):
                shape = arr.shape
                ten = torch.from_numpy(arr.reshape(-1, 1, shape[-2], shape[-1]))
                ten = F.interpolate(
                    ten,
                    size=(input_h, input_w),
                    mode='bilinear',
                    align_corners=False,
                )
                return ten.reshape(
                    shape[0], shape[1], shape[2], input_h, input_w
                ).numpy()

            mci_np = _resize_pcmci(mci_np)
            qval_np = _resize_pcmci(qval_np)
            signif_np = _resize_pcmci(signif_np)
            pcmci_h, pcmci_w = input_h, input_w

        if pcmci_use_abs:
            mci_np = np.abs(mci_np)

        # m1 固定无参数相加场：
        # (E,C,L,H,W) --event均值--> (C,L,H,W)
        #                 --转置--> (L,C,H,W)
        # 该数组随后作为固定 buffer 与 vx 的每个时间步直接相加。
        # m1 不使用 qval/signif，也不使用任何显著性 gate。
        pcmci_m1_add_np = np.mean(
            mci_np,
            axis=0,
            dtype=np.float32,
        ).transpose(1, 0, 2, 3).astype(
            np.float32,
            copy=False,
        )

        # m4 仍使用 [mci, qval, signif] 三个连续输入场。
        # 不把不显著 mci 置 0；signif 只是 m4 可学习使用的输入场。
        pcmci_fields_np = np.stack(
            [mci_np, qval_np, signif_np], axis=0
        ).astype(np.float32, copy=False)  # (3,E,C,L,H,W)

        pcmci_shape_info = {
            'fields': 3,
            'event_type': event_num,
            'predictor': pcmci_predictors,
            'lag': pcmci_lags,
            'height': pcmci_h,
            'width': pcmci_w,
        }

        print('PCMCI shape：', tuple(mci_np.shape))
        print('PCMCI lag 顺序处理：', pcmci_lag_order)
        if pcmci_mode == 'm1':
            print(
                'm1融合：原始MCI在event_type维取均值后，'
                '与vx按lag/predictor/grid固定直接相加。'
            )
            print(
                'm1不使用qval/signif、PCMCI Encoder、gate、'
                'LayerNorm、affine/residual或其他可学习融合参数。'
            )
        else:
            print('PCMCI field 顺序：[mci, qval, signif]')
            print('PCMCI trainable mode：', pcmci_trainable_mode)
            print(
                '说明：pcmci_q_th 在m4局地连续显著性版本中不做硬阈值过滤；'
                '保留该参数仅用于兼容旧调用。'
            )

    # ============================================================
    # 3. 训练/测试划分
    # ============================================================
    all_indices = np.arange(n_seq)

    if ifrandom_split == 'yes':
        train_indices, test_indices = train_test_split(
            all_indices,
            test_size=test_size,
            random_state=random_state,
            shuffle=True,
        )
    elif ifrandom_split == 'no':
        split_index = int((1.0 - test_size) * n_seq)
        train_indices = all_indices[:split_index]
        test_indices = all_indices[split_index:]
    elif ifrandom_split in {'all_train', 'all_test', 'just_model'}:
        train_indices = all_indices
        test_indices = all_indices
    else:
        raise ValueError(
            "ifrandom_split 仅支持 'yes'、'no'、'all_train'、"
            "'all_test' 或 'just_model'。"
        )

    trainx = vxs[train_indices]
    trainy = vys[train_indices]
    testx = vxs[test_indices]
    testy = vys[test_indices]

    # ============================================================
    # 4. 网络组件
    # ============================================================
    def _activation(name):
        name = str(name).lower()
        mapping = {
            'relu': nn.ReLU,
            'leakyrelu': nn.LeakyReLU,
            'prelu': nn.PReLU,
            'sigmoid': nn.Sigmoid,
            'tanh': nn.Tanh,
            'elu': nn.ELU,
            'softmax': lambda: nn.Softmax(dim=1),
        }
        if name not in mapping:
            raise ValueError(f"不支持的激活函数：{name!r}")
        return mapping[name]()

    def _norm_layer(channels):
        if ifnormalization != 'yes':
            return nn.Identity()
        if normalization_method == 'batchnormalization':
            return nn.BatchNorm2d(channels)
        if normalization_method == 'layernormalization':
            # NCHW 下用 GroupNorm(1,C) 实现逐样本、跨通道/空间归一化，
            # 比直接 LayerNorm(C) 更适合卷积特征。
            return nn.GroupNorm(1, channels)
        raise ValueError(
            "normalization_method 必须为 'batchnormalization' 或 "
            "'layernormalization'。"
        )

    class ConvBlock(nn.Module):
        def __init__(self, in_channels, out_channels):
            super().__init__()
            layers = []
            current = in_channels
            padding = cov_kernelsize // 2
            for _ in range(covnum):
                layers.extend([
                    nn.Conv2d(
                        current,
                        out_channels,
                        kernel_size=cov_kernelsize,
                        stride=1,
                        padding=padding,
                    ),
                    _activation(activate),
                    _norm_layer(out_channels),
                ])
                current = out_channels
            self.block = nn.Sequential(*layers)

        def forward(self, x):
            return self.block(x)

    class SpatialEncoder(nn.Module):
        """vx 与 PCMCI 共用相同结构，但参数相互独立。"""

        def __init__(self, in_channels, channel_list):
            super().__init__()
            self.blocks = nn.ModuleList()
            current = in_channels
            for out_ch in channel_list:
                self.blocks.append(ConvBlock(current, out_ch))
                current = out_ch

            if pool_method == 'maxpooling':
                self.pool = nn.MaxPool2d(
                    kernel_size=pool_kernel_size,
                    stride=pool_strides,
                )
            elif pool_method == 'avepooling':
                self.pool = nn.AvgPool2d(
                    kernel_size=pool_kernel_size,
                    stride=pool_strides,
                )
            else:
                raise ValueError(
                    "pool_method 仅支持 'maxpooling' 或 'avepooling'。"
                )

        def forward(self, x):
            features = []
            for stage, block in enumerate(self.blocks):
                x = block(x)
                features.append(x)
                if stage < len(self.blocks) - 1:
                    x = self.pool(x)
            return features

    class DecoderBlock(nn.Module):
        def __init__(self, in_channels, skip_channels, out_channels):
            super().__init__()
            self.up_projection = nn.Conv2d(in_channels, out_channels, 1)
            self.conv = ConvBlock(out_channels + skip_channels, out_channels)

        def forward(self, x, skip):
            x = F.interpolate(
                x,
                size=skip.shape[-2:],
                mode='bilinear',
                align_corners=False,
            )
            x = self.up_projection(x)
            x = torch.cat([x, skip], dim=1)
            return self.conv(x)

    channel_list = [
        int(round(baselayer * (layer_time ** stage)))
        for stage in range(downnum)
    ]
    if any(ch < 1 for ch in channel_list):
        raise ValueError(f"非法 channel_list={channel_list}")

    class STTransUNetPCMCI(nn.Module):
        def __init__(self):
            super().__init__()
            self.timestep = timestep
            self.input_channels = input_channels
            self.input_h = input_h
            self.input_w = input_w
            self.channel_list = channel_list
            self.use_pcmci = use_pcmci
            self.pcmci_mode = pcmci_mode
            self.pcmci_direct_add = bool(self.use_pcmci and pcmci_mode == 'm1')
            self.pcmci_trainable_mode = pcmci_trainable_mode
            self.pcmci_reg_lambda = float(pcmci_reg_lambda)
            self.pcmci_store_diagnostics = bool(pcmci_store_diagnostics)

            self.x_encoder = SpatialEncoder(input_channels, channel_list)

            # 用 dummy 精确确定各层空间尺寸，避免奇数 H/W 手工计算偏差。
            with torch.no_grad():
                dummy = torch.zeros(1, input_channels, input_h, input_w)
                dummy_features = self.x_encoder(dummy)
            self.stage_hw = [tuple(f.shape[-2:]) for f in dummy_features]
            bottleneck_h, bottleneck_w = self.stage_hw[-1]
            bottleneck_channels = channel_list[-1]
            self.bottleneck_h = bottleneck_h
            self.bottleneck_w = bottleneck_w
            self.bottleneck_channels = bottleneck_channels

            if self.use_pcmci:
                fields, events, predictors, lags, ph, pw = pcmci_fields_np.shape
                if lags != timestep:
                    raise RuntimeError("内部错误：PCMCI lag 与 timestep 未对齐。")

                self.pcmci_fields = fields
                self.pcmci_events = events
                self.pcmci_predictors = predictors
                self.pcmci_lags = lags

                if self.pcmci_direct_add:
                    # ------------------------------------------------
                    # m1：严格无门控、无额外可学习参数的固定相加。
                    # ------------------------------------------------
                    # pcmci_m1_add: (T,C,H,W)，与模型输入逐 lag、逐 predictor、
                    # 逐格点一一对应。这里只注册为 buffer，不参与梯度更新。
                    if pcmci_m1_add_np is None:
                        raise RuntimeError("内部错误：m1固定相加场尚未建立。")

                    self.register_buffer(
                        'pcmci_m1_add',
                        torch.from_numpy(pcmci_m1_add_np),
                        persistent=True,
                    )

                    # m1 不建立以下任何额外 PCMCI 可学习模块。
                    self.register_buffer('pcmci_raw', None, persistent=False)
                    self.register_parameter('pcmci_scale', None)
                    self.register_parameter('pcmci_bias', None)
                    self.register_parameter('pcmci_residual', None)
                    self.pcmci_encoder = None
                    self.register_parameter('x_time_embedding', None)
                    self.register_parameter('pcmci_lag_embedding', None)
                    self.prior_gate = None
                    self.local_pcmci_attention = None
                    self.fusion_gate = None
                    self.local_fusion_norm = None

                else:
                    # ------------------------------------------------
                    # m4：保持原局地 Cross-Attention 融合结构。
                    # ------------------------------------------------
                    self.register_buffer(
                        'pcmci_raw',
                        torch.from_numpy(pcmci_fields_np),
                        persistent=True,
                    )
                    self.register_buffer(
                        'pcmci_m1_add',
                        None,
                        persistent=False,
                    )

                    valid_trainable_modes = {
                        'encoder', 'affine', 'affine_residual'
                    }
                    if pcmci_trainable_mode not in valid_trainable_modes:
                        raise ValueError(
                            "pcmci_trainable_mode 必须为 'encoder'、'affine' "
                            "或 'affine_residual'。"
                        )

                    # scale/bias 按 field/event/predictor/lag 学习。
                    calibration_shape = (
                        fields,
                        events,
                        predictors,
                        lags,
                        1,
                        1,
                    )
                    if pcmci_trainable_mode in {'affine', 'affine_residual'}:
                        self.pcmci_scale = nn.Parameter(
                            torch.ones(calibration_shape, dtype=torch.float32)
                        )
                        self.pcmci_bias = nn.Parameter(
                            torch.zeros(calibration_shape, dtype=torch.float32)
                        )
                    else:
                        self.register_parameter('pcmci_scale', None)
                        self.register_parameter('pcmci_bias', None)

                    if pcmci_trainable_mode == 'affine_residual':
                        self.pcmci_residual = nn.Parameter(
                            torch.zeros_like(self.pcmci_raw)
                        )
                    else:
                        self.register_parameter('pcmci_residual', None)

                    pcmci_in_channels = fields * events * predictors
                    self.pcmci_encoder = SpatialEncoder(
                        pcmci_in_channels,
                        channel_list,
                    )

                    self.x_time_embedding = nn.Parameter(
                        torch.zeros(1, timestep, bottleneck_channels)
                    )
                    self.pcmci_lag_embedding = nn.Parameter(
                        torch.zeros(1, lags, bottleneck_channels)
                    )
                    nn.init.trunc_normal_(self.x_time_embedding, std=0.02)
                    nn.init.trunc_normal_(self.pcmci_lag_embedding, std=0.02)

                    gate_channels = (
                        bottleneck_channels if pcmci_gate_per_channel else 1
                    )

                    self.prior_gate = nn.Sequential(
                        nn.Conv2d(
                            bottleneck_channels * 2,
                            bottleneck_channels,
                            kernel_size=3,
                            padding=1,
                        ),
                        nn.ReLU(),
                        nn.Conv2d(
                            bottleneck_channels,
                            gate_channels,
                            kernel_size=1,
                        ),
                        nn.Sigmoid(),
                    )

                    if bottleneck_channels % num_heads != 0:
                        raise ValueError(
                            "局地 Cross-Attention 要求 bottleneck channel 可被 "
                            f"num_heads 整除：{bottleneck_channels} % {num_heads} != 0"
                        )
                    self.local_pcmci_attention = nn.MultiheadAttention(
                        embed_dim=bottleneck_channels,
                        num_heads=num_heads,
                        dropout=trans_dropout_rate,
                        batch_first=True,
                    )

                    self.fusion_gate = nn.Sequential(
                        nn.Conv2d(
                            bottleneck_channels * 2,
                            bottleneck_channels,
                            kernel_size=3,
                            padding=1,
                        ),
                        nn.ReLU(),
                        nn.Conv2d(
                            bottleneck_channels,
                            gate_channels,
                            kernel_size=1,
                        ),
                        nn.Sigmoid(),
                    )
                    self.local_fusion_norm = nn.LayerNorm(
                        bottleneck_channels
                    )

            # 时间 Transformer。推荐 local：每个 bottleneck 格点独立
            # 沿时间维建模，既保持空间对应，也避免旧版 C*H*W 的超大 d_model。
            if temporal_transformer_mode not in {'local', 'global_flatten'}:
                raise ValueError(
                    "temporal_transformer_mode 仅支持 'local' 或 "
                    "'global_flatten'。"
                )
            self.temporal_transformer_mode = temporal_transformer_mode
            if temporal_transformer_mode == 'local':
                temporal_d_model = bottleneck_channels
            else:
                temporal_d_model = (
                    bottleneck_channels * bottleneck_h * bottleneck_w
                )
            if temporal_d_model % num_heads != 0:
                raise ValueError(
                    "Temporal Transformer 的 d_model 必须可被 num_heads 整除："
                    f"{temporal_d_model} % {num_heads} != 0"
                )
            temporal_layer = nn.TransformerEncoderLayer(
                d_model=temporal_d_model,
                nhead=num_heads,
                dim_feedforward=dim_feedforward,
                dropout=trans_dropout_rate,
                activation=nlp_activate,
                batch_first=True,
            )
            self.temporal_transformer = nn.TransformerEncoder(
                temporal_layer,
                num_layers=trans_num,
            )

            decoder_blocks = []
            current_channels = channel_list[-1]
            for stage in range(downnum - 2, -1, -1):
                skip_channels = channel_list[stage]
                decoder_blocks.append(
                    DecoderBlock(
                        current_channels,
                        skip_channels,
                        skip_channels,
                    )
                )
                current_channels = skip_channels
            self.decoder_blocks = nn.ModuleList(decoder_blocks)

            self.output_conv = nn.Conv2d(
                current_channels,
                output_channels,
                kernel_size=1,
            )
            self.output_activation = (
                nn.Identity()
                if if_last_act == 'no'
                else _activation(if_last_act)
            )

            self._last_pcmci_diagnostics = None

        def _calibrated_pcmci(self):
            """仅供 m4 使用，返回校准后的 (3,E,C,L,H,W)。"""
            if self.pcmci_direct_add:
                raise RuntimeError("m1不使用可训练PCMCI校准。")

            prior = self.pcmci_raw
            if self.pcmci_scale is not None:
                prior = prior * self.pcmci_scale + self.pcmci_bias
            if self.pcmci_residual is not None:
                prior = prior + self.pcmci_residual
            return prior

        def pcmci_regularization_loss(self):
            # m1的固定相加分支没有任何PCMCI可学习校准参数，
            # 因此其正则项严格为0。
            if not self.use_pcmci or self.pcmci_direct_add:
                return next(self.parameters()).new_zeros(())

            reg = self.pcmci_raw.new_zeros(())
            if self.pcmci_scale is not None:
                reg = reg + torch.mean((self.pcmci_scale - 1.0) ** 2)
                reg = reg + torch.mean(self.pcmci_bias ** 2)
            if self.pcmci_residual is not None:
                reg = reg + torch.mean(self.pcmci_residual ** 2)
            return reg

        def get_pcmci_diagnostics(self):
            """
            返回最近一次 forward 的诊断结果：
            prior_gate    : 仅m4有效；m1为None
            fusion_gate   : 仅m4有效；m1为None
            attention     : 仅m4有效；m1为None
            direct_add_map: m1实际固定相加的(T,C,H,W)场；m4为None
            """
            return self._last_pcmci_diagnostics

        def forward(self, x):
            if x.ndim != 5:
                raise ValueError(
                    f"模型输入必须为 (B,T,C,H,W)，当前为 {tuple(x.shape)}"
                )
            batch, time_steps, channels, height, width = x.shape
            if time_steps != self.timestep:
                raise ValueError(
                    f"输入 T={time_steps} 与模型 timestep={self.timestep} 不一致"
                )
            if channels != self.input_channels:
                raise ValueError(
                    f"输入 C={channels} 与模型 C={self.input_channels} 不一致"
                )
            if (height, width) != (self.input_h, self.input_w):
                raise ValueError(
                    f"输入 H/W={(height, width)} 与模型 "
                    f"{(self.input_h, self.input_w)} 不一致"
                )

            # ---------------- m1固定无参数相加 ----------------
            if self.use_pcmci and self.pcmci_direct_add:
                if self.pcmci_m1_add.shape != (
                    time_steps,
                    channels,
                    height,
                    width,
                ):
                    raise RuntimeError(
                        "m1固定PCMCI场与模型输入未严格对应："
                        f"pcmci={tuple(self.pcmci_m1_add.shape)}, "
                        f"x={(time_steps, channels, height, width)}"
                    )

                # fixed_pcmci只作为buffer广播到batch维，不参与梯度更新。
                x = x + self.pcmci_m1_add.unsqueeze(0)

                if self.pcmci_store_diagnostics:
                    self._last_pcmci_diagnostics = {
                        'prior_gate': None,
                        'fusion_gate': None,
                        'attention': None,
                        'direct_add_map': (
                            self.pcmci_m1_add.detach().cpu().numpy()
                        ),
                    }

            # ---------------- vx Encoder ----------------
            x_flat = x.reshape(
                batch * time_steps,
                channels,
                height,
                width,
            )
            x_features_flat = self.x_encoder(x_flat)
            x_features = []
            for feat in x_features_flat:
                _, c, h, w = feat.shape
                x_features.append(
                    feat.reshape(batch, time_steps, c, h, w)
                )

            x_bottleneck = x_features[-1]
            b, t, c, hb, wb = x_bottleneck.shape

            # ---------------- m4 PCMCI局地融合 ----------------
            # m1已经在输入层完成固定无参数相加，此处不再进入任何PCMCI模块。
            if self.use_pcmci and not self.pcmci_direct_add:
                prior = self._calibrated_pcmci()

                # (F,E,C,L,H,W) -> (L,F*E*C,H,W)
                prior_lag = prior.permute(3, 0, 1, 2, 4, 5).contiguous()
                prior_lag = prior_lag.reshape(
                    self.pcmci_lags,
                    self.pcmci_fields
                    * self.pcmci_events
                    * self.pcmci_predictors,
                    self.input_h,
                    self.input_w,
                )

                # 每个lag作为样本通过参数独立的PCMCI Encoder。
                pcmci_features = self.pcmci_encoder(prior_lag)
                pcmci_bottleneck_single = pcmci_features[-1]
                pcmci_bottleneck = pcmci_bottleneck_single.unsqueeze(0).expand(
                    batch,
                    -1,
                    -1,
                    -1,
                    -1,
                )

                if pcmci_bottleneck.shape[1:] != x_bottleneck.shape[1:]:
                    raise RuntimeError(
                        "vx与PCMCI bottleneck未严格对应："
                        f"vx={tuple(x_bottleneck.shape)}, "
                        f"pcmci={tuple(pcmci_bottleneck.shape)}"
                    )

                gate_input = torch.cat(
                    [x_bottleneck, pcmci_bottleneck],
                    dim=2,
                ).reshape(
                    b * t,
                    2 * c,
                    hb,
                    wb,
                )
                prior_gate = self.prior_gate(gate_input).reshape(
                    b,
                    t,
                    -1,
                    hb,
                    wb,
                )
                pcmci_gated = pcmci_bottleneck * prior_gate

                # 每个格点独立做局地Cross-Attention。
                x_local = x_bottleneck.permute(
                    0,
                    3,
                    4,
                    1,
                    2,
                ).reshape(
                    b * hb * wb,
                    t,
                    c,
                )
                p_local = pcmci_gated.permute(
                    0,
                    3,
                    4,
                    1,
                    2,
                ).reshape(
                    b * hb * wb,
                    t,
                    c,
                )

                x_local = x_local + self.x_time_embedding
                p_local = p_local + self.pcmci_lag_embedding

                need_weights = self.pcmci_store_diagnostics
                attention_out, attention_weights = self.local_pcmci_attention(
                    query=x_local,
                    key=p_local,
                    value=p_local,
                    need_weights=need_weights,
                    average_attn_weights=True,
                )
                fusion_source = attention_out.reshape(
                    b,
                    hb,
                    wb,
                    t,
                    c,
                ).permute(
                    0,
                    3,
                    4,
                    1,
                    2,
                ).contiguous()

                fusion_input = torch.cat(
                    [x_bottleneck, fusion_source],
                    dim=2,
                ).reshape(
                    b * t,
                    2 * c,
                    hb,
                    wb,
                )
                fusion_gate = self.fusion_gate(fusion_input).reshape(
                    b,
                    t,
                    -1,
                    hb,
                    wb,
                )

                fused = x_bottleneck + fusion_gate * fusion_source

                fused_local = fused.permute(
                    0,
                    3,
                    4,
                    1,
                    2,
                ).reshape(
                    b * hb * wb,
                    t,
                    c,
                )
                fused_local = self.local_fusion_norm(fused_local)
                x_bottleneck = fused_local.reshape(
                    b,
                    hb,
                    wb,
                    t,
                    c,
                ).permute(
                    0,
                    3,
                    4,
                    1,
                    2,
                ).contiguous()

                if self.pcmci_store_diagnostics:
                    if prior_gate.shape[2] == 1:
                        pg = prior_gate.mean(dim=0).squeeze(1)
                    else:
                        pg = prior_gate.mean(dim=0)

                    if fusion_gate.shape[2] == 1:
                        fg = fusion_gate.mean(dim=0).squeeze(1)
                    else:
                        fg = fusion_gate.mean(dim=0)

                    # attention_weights: (B*Hb*Wb,T,L)
                    aw = attention_weights.reshape(
                        b,
                        hb,
                        wb,
                        t,
                        self.pcmci_lags,
                    ).permute(
                        0,
                        3,
                        4,
                        1,
                        2,
                    )
                    aw = aw.mean(dim=0)

                    self._last_pcmci_diagnostics = {
                        'prior_gate': pg.detach().cpu().numpy(),
                        'fusion_gate': fg.detach().cpu().numpy(),
                        'attention': aw.detach().cpu().numpy(),
                        'direct_add_map': None,
                    }

            # ---------------- Temporal Transformer ----------------
            if self.temporal_transformer_mode == 'local':
                # 每个格点一个独立时间序列：(B*Hb*Wb,T,Cb)
                temporal_seq = x_bottleneck.permute(
                    0, 3, 4, 1, 2
                ).reshape(b * hb * wb, t, c)
                temporal_out = self.temporal_transformer(temporal_seq)
                temporal_out = temporal_out.reshape(
                    b, hb, wb, t, c
                ).permute(0, 3, 4, 1, 2).contiguous()
            else:
                # 旧版兼容路径：(B,T,Cb*Hb*Wb)
                temporal_seq = x_bottleneck.reshape(b, t, -1)
                temporal_out = self.temporal_transformer(temporal_seq)
                temporal_out = temporal_out.reshape(b, t, c, hb, wb)

            # 与原版本一致：取最后一个时间步进入 Decoder。
            decoded = temporal_out[:, -1]

            # 浅层 skip 同样取最后一个时间步。
            skip_features = [feat[:, -1] for feat in x_features[:-1]]
            for decoder_block, skip in zip(
                self.decoder_blocks,
                reversed(skip_features),
            ):
                decoded = decoder_block(decoded, skip)

            output = self.output_conv(decoded)
            output = self.output_activation(output)
            return output

    # ============================================================
    # 5. 损失、指标、优化器
    # ============================================================
    def _pearson_loss(y_pred, y_true):
        # 每个空间/通道沿 batch 计算相关，再平均。
        pred_mean = torch.mean(y_pred, dim=0, keepdim=True)
        true_mean = torch.mean(y_true, dim=0, keepdim=True)
        pred_dev = y_pred - pred_mean
        true_dev = y_true - true_mean
        numerator = torch.sum(pred_dev * true_dev, dim=0)
        denominator = torch.sqrt(
            torch.sum(pred_dev ** 2, dim=0)
            * torch.sum(true_dev ** 2, dim=0)
        ).clamp_min(1e-12)
        corr = (numerator / denominator).clamp(-1.0, 1.0)
        return torch.mean((1.0 - corr) ** 1.5)

    def _make_loss():
        if not isinstance(loss_function, str):
            return loss_function
        name = loss_function.lower()
        if task_mode == 'regression':
            mapping = {
                'default': nn.MSELoss,
                'mseloss': nn.MSELoss,
                'l1loss': nn.L1Loss,
                'poissonnllloss': nn.PoissonNLLLoss,
                'gaussiannllloss': nn.GaussianNLLLoss,
                'kldivloss': nn.KLDivLoss,
                'huberloss': nn.HuberLoss,
                'smoothl1loss': nn.SmoothL1Loss,
            }
            if name == 'pearsonr':
                return _pearson_loss
            if name not in mapping:
                raise ValueError(f"不支持的 regression loss：{loss_function}")
            return mapping[name]()
        if task_mode == 'binary_classify':
            if name in {'default', 'bcewithlogitsloss'} and if_last_act == 'no':
                return nn.BCEWithLogitsLoss()
            if name in {'default', 'bceloss'}:
                return nn.BCELoss()
            if name == 'softmarginloss':
                return nn.SoftMarginLoss()
            if name == 'multilabelsoftmarginloss':
                return nn.MultiLabelSoftMarginLoss()
            raise ValueError(f"不支持的 binary loss：{loss_function}")
        # multi_classify
        if name in {'default', 'crossentropyloss'}:
            return nn.CrossEntropyLoss(ignore_index=-100)
        if name == 'nllloss':
            return nn.NLLLoss(ignore_index=-100)
        raise ValueError(f"不支持的 multiclass loss：{loss_function}")

    criterion = _make_loss()

    def _metric_value(y_pred, y_true):
        if not isinstance(metrics, str):
            return metrics(y_pred, y_true)
        name = metrics.lower()
        if task_mode == 'regression':
            if name in {'default', 'mseloss'}:
                return F.mse_loss(y_pred, y_true)
            if name == 'l1loss':
                return F.l1_loss(y_pred, y_true)
            if name == 'pearsonr':
                return _pearson_loss(y_pred, y_true)
            if name == 'huberloss':
                return F.huber_loss(y_pred, y_true)
            if name == 'smoothl1loss':
                return F.smooth_l1_loss(y_pred, y_true)
            raise ValueError(f"不支持的 regression metrics：{metrics}")

        if task_mode == 'binary_classify':
            prob = torch.sigmoid(y_pred) if if_last_act == 'no' else y_pred
            pred_label = (prob >= 0.5).long()
            true_label = y_true.long()
            if name in {'default', 'f1'}:
                tp = torch.sum((pred_label == 1) & (true_label == 1)).float()
                fp = torch.sum((pred_label == 1) & (true_label == 0)).float()
                fn = torch.sum((pred_label == 0) & (true_label == 1)).float()
                return 2 * tp / (2 * tp + fp + fn).clamp_min(1.0)
            if name == 'accuracy':
                return torch.mean((pred_label == true_label).float())
            if name == 'precision':
                tp = torch.sum((pred_label == 1) & (true_label == 1)).float()
                fp = torch.sum((pred_label == 1) & (true_label == 0)).float()
                return tp / (tp + fp).clamp_min(1.0)
            if name == 'recall':
                tp = torch.sum((pred_label == 1) & (true_label == 1)).float()
                fn = torch.sum((pred_label == 0) & (true_label == 1)).float()
                return tp / (tp + fn).clamp_min(1.0)
            raise ValueError(f"不支持的 binary metrics：{metrics}")

        pred_label = torch.argmax(y_pred, dim=1)
        mask = y_true != -100
        if not torch.any(mask):
            return y_pred.new_tensor(0.0)
        return torch.mean((pred_label[mask] == y_true[mask]).float())

    def _compute_data_loss(output, target):
        if task_mode == 'multi_classify':
            if isinstance(loss_function, str) and loss_function.lower() == 'nllloss':
                return criterion(F.log_softmax(output, dim=1), target.long())
            return criterion(output, target.long())
        return criterion(output, target.float())

    def _make_optimizer(model_instance):
        name = str(optimizer).lower()
        if name == 'sgd':
            return SGD(model_instance.parameters(), lr=learning_rate)
        if name == 'adam':
            return Adam(model_instance.parameters(), lr=learning_rate)
        if name == 'nadam':
            return NAdam(model_instance.parameters(), lr=learning_rate)
        raise ValueError(f"不支持的 optimizer={optimizer!r}")

    def _make_model():
        return STTransUNetPCMCI().to(devices)

    # ============================================================
    # 6. 训练辅助
    # ============================================================
    class EarlyStopping:
        def __init__(self, patience, delta=0.0):
            self.patience = int(patience)
            self.delta = float(delta)
            self.counter = 0
            self.best_loss = np.inf
            self.best_state = None
            self.early_stop = False

        def step(self, value, model_instance):
            if not np.isfinite(value):
                self.early_stop = True
                return
            if value < self.best_loss - self.delta:
                self.best_loss = value
                self.counter = 0
                self.best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in model_instance.state_dict().items()
                }
            else:
                self.counter += 1
                if self.counter >= self.patience:
                    self.early_stop = True

        def restore(self, model_instance):
            if self.best_state is not None:
                model_instance.load_state_dict(self.best_state)

    def _make_loader(x_data, y_data, shuffle):
        x_tensor = torch.from_numpy(np.ascontiguousarray(x_data))
        y_tensor = torch.from_numpy(np.ascontiguousarray(y_data))
        dataset = TensorDataset(x_tensor, y_tensor)
        return DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=(devices.type == 'cuda'),
            drop_last=False,
        )

    def _run_epoch(model_instance, loader, optimizer_instance=None):
        training = optimizer_instance is not None
        model_instance.train(training)
        total_loss = 0.0
        total_metric = 0.0
        total_samples = 0

        grad_context = torch.enable_grad() if training else torch.no_grad()
        with grad_context:
            for batch_x, batch_y in loader:
                batch_x = batch_x.to(
                    devices, non_blocking=True, dtype=torch.float32
                )
                if task_mode == 'multi_classify':
                    batch_y = batch_y.to(
                        devices, non_blocking=True, dtype=torch.long
                    )
                else:
                    batch_y = batch_y.to(
                        devices, non_blocking=True, dtype=torch.float32
                    )

                if training:
                    optimizer_instance.zero_grad(set_to_none=True)

                output = model_instance(batch_x)
                data_loss = _compute_data_loss(output, batch_y)
                reg_loss = model_instance.pcmci_regularization_loss()
                total_batch_loss = data_loss + pcmci_reg_lambda * reg_loss
                metric_value = _metric_value(output, batch_y)

                if training:
                    total_batch_loss.backward()
                    optimizer_instance.step()

                n = batch_x.shape[0]
                total_samples += n
                total_loss += float(total_batch_loss.detach().cpu()) * n
                total_metric += float(metric_value.detach().cpu()) * n

        if total_samples == 0:
            return np.nan, np.nan
        return total_loss / total_samples, total_metric / total_samples

    def _train_one_model(
        model_instance,
        optimizer_instance,
        fit_x,
        fit_y,
        val_x,
        val_y,
    ):
        fit_loader = _make_loader(fit_x, fit_y, shuffle=True)
        val_loader = _make_loader(val_x, val_y, shuffle=False)
        stopper = (
            EarlyStopping(if_early_stopping)
            if if_early_stopping is not None
            else None
        )

        for epoch_index in range(epochs):
            start = datetime.datetime.now()
            train_loss_value, train_metric_value = _run_epoch(
                model_instance,
                fit_loader,
                optimizer_instance,
            )
            val_loss_value, val_metric_value = _run_epoch(
                model_instance,
                val_loader,
                optimizer_instance=None,
            )
            end = datetime.datetime.now()

            if ifmute == 'no':
                print(
                    f"第 {epoch_index + 1} 次训练 "
                    f"loss={train_loss_value:.8g}，"
                    f"metric={train_metric_value:.8g}；"
                    f"验证 loss={val_loss_value:.8g}，"
                    f"metric={val_metric_value:.8g}；"
                    f"用时={end - start}"
                )

            if stopper is not None:
                stopper.step(val_loss_value, model_instance)
                if stopper.early_stop:
                    if ifmute == 'no':
                        print('Early stopping，恢复验证集最优模型。')
                    stopper.restore(model_instance)
                    break

        if stopper is not None and not stopper.early_stop:
            stopper.restore(model_instance)

    def _load_checkpoint(model_instance, optimizer_instance, base_path):
        model_file = base_path + '.pth'
        optimizer_file = base_path + '_opt.pth'
        model_instance.load_state_dict(
            torch.load(model_file, map_location=devices)
        )
        if os.path.exists(optimizer_file):
            optimizer_instance.load_state_dict(
                torch.load(optimizer_file, map_location=devices)
            )
            # optimizer state 也移动到目标 GPU。
            for state in optimizer_instance.state.values():
                for key, value in state.items():
                    if torch.is_tensor(value):
                        state[key] = value.to(devices)

    # ============================================================
    # 7. 建模与训练
    # ============================================================
    models = None
    opts = None

    if k_fold is not None:
        if int(k_fold) < 2:
            raise ValueError("k_fold 必须为 None 或 >=2。")
        models = []
        opts = []
        kf = KFold(
            n_splits=int(k_fold),
            shuffle=True,
            random_state=random_state,
        )
        for fold_no, (fit_idx, val_idx) in enumerate(kf.split(trainx), start=1):
            current_model = _make_model()
            current_opt = _make_optimizer(current_model)
            if if_best_mode != 'no':
                if modelpath is None:
                    raise ValueError("加载模型时 modelpath 不能为空。")
                _load_checkpoint(
                    current_model,
                    current_opt,
                    modelpath + '_' + str(fold_no),
                )
            if if_print_model == 'yes' and fold_no == 1:
                print(current_model)
            if ifrandom_split not in {'all_test', 'just_model'}:
                _train_one_model(
                    current_model,
                    current_opt,
                    trainx[fit_idx],
                    trainy[fit_idx],
                    trainx[val_idx],
                    trainy[val_idx],
                )
            models.append(current_model)
            opts.append(current_opt)
        model = None
    else:
        model = _make_model()
        opt = _make_optimizer(model)
        if if_best_mode != 'no':
            if modelpath is None:
                raise ValueError("加载模型时 modelpath 不能为空。")
            _load_checkpoint(model, opt, modelpath)
        if if_print_model == 'yes':
            print(model)

        if ifrandom_split not in {'all_test', 'just_model'}:
            if valid_size is None or valid_size <= 0:
                fit_x, fit_y = trainx, trainy
                val_x, val_y = testx, testy
            else:
                relative_valid = valid_size / max(1e-12, 1.0 - test_size)
                relative_valid = min(max(relative_valid, 1e-6), 0.999999)
                if ifrandom_split == 'yes':
                    fit_x, val_x, fit_y, val_y = train_test_split(
                        trainx,
                        trainy,
                        test_size=relative_valid,
                        random_state=random_state,
                        shuffle=True,
                    )
                else:
                    val_count = max(1, int(round(len(trainx) * relative_valid)))
                    fit_x, val_x = trainx[:-val_count], trainx[-val_count:]
                    fit_y, val_y = trainy[:-val_count], trainy[-val_count:]
                    if len(fit_x) == 0:
                        raise ValueError(
                            "训练样本过少，无法进一步划分验证集。"
                        )
            _train_one_model(model, opt, fit_x, fit_y, val_x, val_y)

    if ifrandom_split == 'just_model':
        heatmap = 0
        weights = 0
        if ifsave == 'yes':
            if savepath is None:
                raise ValueError("ifsave='yes' 时 savepath 不能为空。")
            if k_fold is None:
                torch.save(model.state_dict(), savepath + '.pth')
                torch.save(opt.state_dict(), savepath + '_opt.pth')
            else:
                for fold_index, (m, o) in enumerate(zip(models, opts), start=1):
                    torch.save(m.state_dict(), f'{savepath}_{fold_index}.pth')
                    torch.save(o.state_dict(), f'{savepath}_{fold_index}_opt.pth')
        return (
            models if k_fold is not None else model,
            None,
            None,
            None,
            None,
            heatmap,
            weights,
        )

    # ============================================================
    # 8. 推理
    # ============================================================
    def _predict_one(model_instance, x_data):
        loader = DataLoader(
            TensorDataset(torch.from_numpy(np.ascontiguousarray(x_data))),
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            pin_memory=(devices.type == 'cuda'),
            drop_last=False,
        )
        outputs = []
        model_instance.eval()
        with torch.no_grad():
            for (batch_x,) in loader:
                batch_x = batch_x.to(
                    devices, non_blocking=True, dtype=torch.float32
                )
                out = model_instance(batch_x)
                if task_mode == 'binary_classify' and if_last_act == 'no':
                    out = torch.sigmoid(out)
                elif task_mode == 'multi_classify':
                    out = torch.softmax(out, dim=1)
                outputs.append(out.detach().cpu().numpy())
        return np.concatenate(outputs, axis=0)

    if k_fold is None:
        predicty = _predict_one(model, testx)
    else:
        fold_predictions = [_predict_one(m, testx) for m in models]
        predicty = np.mean(np.stack(fold_predictions, axis=0), axis=0)

    # ============================================================
    # 9. 回归相关系数 r/p
    # ============================================================
    r = None
    p = None
    if task_mode == 'regression':
        out_c, out_h, out_w = testy.shape[1:]
        r = np.full((out_c, out_h, out_w), np.nan, dtype=np.float32)
        p = np.full((out_c, out_h, out_w), np.nan, dtype=np.float32)
        for c_index in range(out_c):
            for h_index in range(out_h):
                for w_index in range(out_w):
                    true_series = testy[:, c_index, h_index, w_index]
                    pred_series = predicty[:, c_index, h_index, w_index]
                    valid = np.isfinite(true_series) & np.isfinite(pred_series)
                    if np.sum(valid) < 2:
                        continue
                    if (
                        np.nanstd(true_series[valid]) == 0
                        or np.nanstd(pred_series[valid]) == 0
                    ):
                        continue
                    corr, p_value = pearsonr(
                        true_series[valid], pred_series[valid]
                    )
                    r[c_index, h_index, w_index] = corr
                    p[c_index, h_index, w_index] = p_value

    # ============================================================
    # 10. 可选 SHAP / permutation importance
    # ============================================================
    heatmap = 0
    weights = 0

    def _representative_model():
        return model if k_fold is None else models[0]

    if ifheatmap == 'yes' or ifweight in {'yes', 'shap'}:
        try:
            import shap

            sample_count = min(50, len(testx))
            rng = np.random.default_rng(random_state)
            sample_idx = rng.choice(len(testx), size=sample_count, replace=False)
            sample_x_np = testx[sample_idx]
            sample_x = torch.from_numpy(sample_x_np).to(
                devices, dtype=torch.float32
            )

            class ShapWrapper(nn.Module):
                def __init__(self, wrapped_model, output_index):
                    super().__init__()
                    self.wrapped_model = wrapped_model
                    self.output_index = output_index

                def forward(self, x_input):
                    result = self.wrapped_model(x_input)
                    if task_mode == 'multi_classify':
                        result = torch.softmax(result, dim=1)
                    elif task_mode == 'binary_classify' and if_last_act == 'no':
                        result = torch.sigmoid(result)
                    result = result[:, self.output_index]
                    return result.reshape(result.shape[0], -1).mean(dim=1, keepdim=True)

            shap_by_output = []
            selected_models = [model] if k_fold is None else models
            for out_index in range(output_channels):
                fold_values = []
                for current_model in selected_models:
                    current_model.eval()
                    wrapper = ShapWrapper(current_model, out_index).to(devices)
                    explainer = shap.GradientExplainer(wrapper, sample_x)
                    values = explainer.shap_values(sample_x)
                    if isinstance(values, list):
                        values = values[0]
                    values = np.asarray(values)
                    # 某些 SHAP 版本末尾会保留单输出维。
                    if values.ndim == 6 and values.shape[-1] == 1:
                        values = values[..., 0]
                    fold_values.append(np.abs(values))
                shap_by_output.append(
                    np.mean(np.stack(fold_values, axis=0), axis=0)
                )

            shap_values = np.stack(shap_by_output, axis=0)
            # (Cout,N,T,Cin,H,W)
            if ifheatmap == 'yes':
                hm = np.mean(shap_values, axis=(1, 2))  # (Cout,Cin,H,W)
                heatmap = hm.transpose(0, 2, 3, 1)     # (Cout,H,W,Cin)

            if ifweight in {'yes', 'shap'}:
                raw_weight = np.mean(shap_values, axis=(1, 2, 4, 5))
                denominator = np.sum(raw_weight, axis=1, keepdims=True)
                weights = np.divide(
                    raw_weight,
                    denominator,
                    out=np.zeros_like(raw_weight),
                    where=denominator != 0,
                ) * 100.0
                if ifmute == 'no':
                    for out_index in range(weights.shape[0]):
                        for in_index in range(weights.shape[1]):
                            print(
                                f'预报因子 {in_index + 1} 对预报值 '
                                f'{out_index + 1} 的贡献：'
                                f'{weights[out_index, in_index]:.6f} %'
                            )
                        print()
        except Exception as exc:
            warnings.warn(
                f"SHAP 计算失败，heatmap/weights 保持为 0：{exc}",
                RuntimeWarning,
            )

    elif ifweight == 'oob':
        # 通道置换重要性：按输出通道比较整体 MSE 增量。
        repeats = 3
        base_prediction = predicty
        raw_importance = np.zeros(
            (output_channels, input_channels), dtype=np.float64
        )
        rng = np.random.default_rng(random_state)
        for in_index in range(input_channels):
            increases = []
            for _ in range(repeats):
                shuffled = testx.copy()
                perm = rng.permutation(len(shuffled))
                shuffled[:, :, in_index] = shuffled[perm, :, in_index]
                if k_fold is None:
                    shuffled_prediction = _predict_one(model, shuffled)
                else:
                    shuffled_prediction = np.mean(
                        np.stack(
                            [_predict_one(m, shuffled) for m in models],
                            axis=0,
                        ),
                        axis=0,
                    )
                if task_mode == 'regression':
                    base_error = np.mean(
                        (base_prediction - testy) ** 2,
                        axis=(0, 2, 3),
                    )
                    shuffled_error = np.mean(
                        (shuffled_prediction - testy) ** 2,
                        axis=(0, 2, 3),
                    )
                    increases.append(shuffled_error - base_error)
                else:
                    raise NotImplementedError(
                        "当前 oob 置换重要性仅实现 regression。"
                    )
            raw_importance[:, in_index] = np.mean(increases, axis=0)

        denominator = np.sum(raw_importance, axis=1, keepdims=True)
        weights = np.divide(
            raw_importance,
            denominator,
            out=np.zeros_like(raw_importance),
            where=denominator != 0,
        ) * 100.0

    # ============================================================
    # 11. 保存
    # ============================================================
    if ifsave == 'yes':
        if savepath is None:
            raise ValueError("ifsave='yes' 时 savepath 不能为空。")
        save_dir = os.path.dirname(savepath)
        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
        if k_fold is None:
            torch.save(model.state_dict(), savepath + '.pth')
            torch.save(opt.state_dict(), savepath + '_opt.pth')
        else:
            for fold_index, (m, o) in enumerate(zip(models, opts), start=1):
                torch.save(m.state_dict(), f'{savepath}_{fold_index}.pth')
                torch.save(o.state_dict(), f'{savepath}_{fold_index}_opt.pth')

    return models if k_fold is not None else model,predicty,testy,r,p,heatmap,weights,
    
